In [ ]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from sklearn import set_config

set_config(display="diagram")

In [4]:
def load_data(file_path: str):
    """
    Loads a dataset
    """
    df = pd.read_csv(file_path)
    return df


In [ ]:
train_df = load_data("../raw_data/predictive_analytics/train.csv")

test_df = load_data("../raw_data/predictive_analytics/test.csv")

data_descriptions_df = load_data(
    "../raw_data/predictive_analytics/data_descriptions.csv"
)

In [8]:
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Descriptions shape:", data_descriptions_df.shape)

Train shape: (243787, 21)
Test shape: (104480, 20)
Descriptions shape: (21, 4)


In [9]:
train_df.head()

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,...,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled,CustomerID,Churn
0,20,11.055215,221.104302,Premium,Mailed check,No,Both,No,Mobile,36.758104,...,10,Sci-Fi,2.176498,4,Male,3,No,No,CB6SXPNVZA,0
1,57,5.175208,294.986882,Basic,Credit card,Yes,Movies,No,Tablet,32.450568,...,18,Action,3.478632,8,Male,23,No,Yes,S7R2G87O09,0
2,73,12.106657,883.785952,Basic,Mailed check,Yes,Movies,No,Computer,7.395160,...,23,Fantasy,4.238824,6,Male,1,Yes,Yes,EASDC20BDT,0
3,32,7.263743,232.439774,Basic,Electronic check,No,TV Shows,No,Tablet,27.960389,...,30,Drama,4.276013,2,Male,24,Yes,Yes,NPF69NT69N,0
4,57,16.953078,966.325422,Premium,Electronic check,Yes,TV Shows,No,TV,20.083397,...,20,Comedy,3.616170,4,Female,0,No,No,4LGYPK7VOL,0


In [10]:
test_df.head()

,AccountAge,MonthlyCharges,TotalCharges,SubscriptionType,PaymentMethod,PaperlessBilling,ContentType,MultiDeviceAccess,DeviceRegistered,ViewingHoursPerWeek,AverageViewingDuration,ContentDownloadsPerMonth,GenrePreference,UserRating,SupportTicketsPerMonth,Gender,WatchlistSize,ParentalControl,SubtitlesEnabled,CustomerID
0,38,17.869374,679.036195,Premium,Mailed check,No,TV Shows,No,TV,29.126308,122.274031,42,Comedy,3.522724,2,Male,23,No,No,O1W6BHP6RM
1,77,9.912854,763.289768,Basic,Electronic check,Yes,TV Shows,No,TV,36.873729,57.093319,43,Action,2.021545,2,Female,22,Yes,No,LFR4X92X8H
2,5,15.019011,75.095057,Standard,Bank transfer,No,TV Shows,Yes,Computer,7.601729,140.414001,14,Sci-Fi,4.806126,2,Female,22,No,Yes,QM5GBIYODA
3,88,15.357406,1351.451692,Standard,Electronic check,No,Both,Yes,Tablet,35.586430,177.002419,14,Comedy,4.943900,0,Female,23,Yes,Yes,D9RXTK2K9F
4,91,12.406033,1128.949004,Standard,Credit card,Yes,TV Shows,Yes,Tablet,23.503651,70.308376,6,Drama,2.846880,6,Female,0,No,No,ENTCCHR1LR


In [12]:
data_descriptions_df.head()

,Column_name,Column_type,Data_type,Description
0,AccountAge,Feature,integer,The age of the user's account in months.
1,MonthlyCharges,Feature,float,The amount charged to the user on a monthly ba...
2,TotalCharges,Feature,float,The total charges incurred by the user over th...
3,SubscriptionType,Feature,object,The type of subscription chosen by the user (B...
4,PaymentMethod,Feature,string,The method of payment used by the user.


In [ ]:
def transform_columns(df: pd.DataFrame):
    df = df.copy()

    clean_cols = df.columns.str.strip().str.replace(' ', '_')

    # Insert underscore between lowercase letters/digits and an uppercase letter (e.g., CustomerID -> Customer_ID)
    clean_cols = clean_cols.str.replace(r'([a-z0-9])([A-Z])', r'\1_\2', regex=True)

    # Insert underscore between consecutive uppercase letters followed by lowercase (e.g., USAUser -> USA_User)
    clean_cols = clean_cols.str.replace(r'([A-Z]+)([A-Z][a-z])', r'\1_\2', regex=True)

    # Lowercase everything and collapse any sequential underscores (e.g., customer__id -> customer_id)
    df.columns = clean_cols.str.lower().str.replace(r'_+', '_', regex=True)
    return df


In [ ]:
def preprocess_data(df: pd.DataFrame):
    """
    Prepares the feature preprocessing pipeline.
    """
    # 1. Clean column names: Strip whitespace and replace spaces with underscores
    df = transform_columns(df)
    # 2. Separate target from features
    # (Assuming the target column name becomes 'churn' after formatting)
    X = df.drop(columns=['churn', 'customer_id'], errors='ignore')
    y = df['churn']

    # 3. Identify feature types automatically or explicitly
    numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

    # 4. Define transformations per data type (Best Practice)
    # Numerical: Impute missing values with median, then scale
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    # Categorical: Impute missing values with most frequent string, then One-Hot Encode
    categorical_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # 5. Bundle transformers into a ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numeric_features),
            ('cat', categorical_transformer, categorical_features)
        ],
        remainder='drop', # Explicitly drop columns we didn't specify (like IDs)
        verbose_feature_names_out=False
    ).set_output(transform='pandas')

    # 6. Train/Test Split BEFORE fitting anything to avoid data leakage
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42
    )

    return X_train, X_test, y_train, y_test, preprocessor

In [37]:
if __name__ == "__main__":
    # Example execution within your Docker container volume path
    data_descriptions_df = load_data('../raw_data/data_descriptions.csv')
    data_df = load_data('../raw_data/train.csv')
    #data_test_df = load_data('raw_data/test.csv')

    print(f"data_descriptions_df shape: {data_descriptions_df.shape}")
    print(f"data_df shape: {data_df.shape}")
    print(f"")

    X_train, X_test, y_train, y_test, preprocessor  = preprocess_data(data_df)

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")

data_descriptions_df shape: (21, 4)
data_df shape: (243787, 21)

X_train shape: (195029, 19)
y_train shape: (195029,)

X_test shape: (48758, 19)
y_test shape: (48758,)


In [39]:
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,False
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [40]:
set_config(display='text')
preprocessor.get_params()

{'force_int_remainder_cols': 'deprecated',
 'n_jobs': None,
 'remainder': 'drop',
 'sparse_threshold': 0.3,
 'transformer_weights': None,
 'transformers': [('num',
   Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                   ('scaler', StandardScaler())]),
   ['account_age',
    'monthly_charges',
    'total_charges',
    'viewing_hours_per_week',
    'average_viewing_duration',
    'content_downloads_per_month',
    'user_rating',
    'support_tickets_per_month',
    'watchlist_size']),
  ('cat',
   Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                   ('onehot',
                    OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
   ['subscription_type',
    'payment_method',
    'paperless_billing',
    'content_type',
    'multi_device_access',
    'device_registered',
    'genre_preference',
    'gender',
    'parental_control',
    'subtitles_enabled'])],
 'verbose': False,
 'verbose_feature_names_out': Fal

In [46]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_processed = pd.DataFrame(
    X_train_transformed
)
X_processed.head(3)

,account_age,monthly_charges,total_charges,viewing_hours_per_week,average_viewing_duration,content_downloads_per_month,user_rating,support_tickets_per_month,watchlist_size,subscription_type_Basic,...,genre_preference_Comedy,genre_preference_Drama,genre_preference_Fantasy,genre_preference_Sci-Fi,gender_Female,gender_Male,parental_control_No,parental_control_Yes,subtitles_enabled_No,subtitles_enabled_Yes
112198,-0.175398,1.426607,0.493176,-0.234045,1.564359,-1.143216,-1.227746,-0.175521,0.692354,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
35340,-0.671295,0.196020,-0.490383,1.389123,1.636791,0.728099,0.688829,-1.220740,0.414380,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0
33223,0.495521,-0.242718,0.250187,1.686873,-0.519319,-0.103596,-0.971997,-0.523927,0.136405,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0


In [47]:
X_processed.shape

(195029, 38)